# Analysis and plotting methods for manuscript figures

This notebook contains selected code used to prepare figures on funding composition, differences across disease areas, and research orientation. Where original plotting code embedded regression estimates as literal values, that plotting cell was omitted. Numerical results are reported in the manuscript.


## Figure 1: funding over time and across disease areas

Calculate annual funding shares and disease-area funding composition, then build the two-panel display. Publications can appear in more than one disease area.


In [ ]:
df_0 = pd.read_json('a5_data_final_v3.json', lines=True)

In [ ]:
df_0['year'] = df_0['year'].astype(int)
df_1 = df_0[df_0['year'] >= 2009]

In [ ]:
mesh_lvl1_to_name = {
    "C01": "Infections",
    "C04": "Neoplasms",
    "C05": "Musculoskeletal Diseases",
    "C06": "Digestive System Diseases",
    "C07": "Stomatognathic Diseases",
    "C08": "Respiratory Tract Diseases",
    "C09": "Otorhinolaryngologic Diseases",
    "C10": "Nervous System Diseases",
    "C11": "Eye Diseases",
    "C12": "Urogenital Diseases",
    "C14": "Cardiovascular Diseases",
    "C15": "Hemic and Lymphatic Diseases",
    "C16": "Congenital, Hereditary, and Neonatal Diseases and Abnormalities",
    "C17": "Skin and Connective Tissue Diseases",
    "C18": "Nutritional and Metabolic Diseases",
    "C19": "Endocrine System Diseases",
    "C20": "Immune System Diseases",
    "C21": "Disorders of Environmental Origin",
    "C22": "Animal Diseases",
    "C23": "Pathological Conditions, Signs and Symptoms",
    "C24": "Occupational Diseases",
    "C25": "Chemically-Induced Disorders",
    "C26": "Wounds and Injuries",
    "F03": "Mental Disorders"
}

df_mesh_lvl1 = pd.DataFrame([
    {"mesh_lvl1": k, "category_name": v}
    for k, v in mesh_lvl1_to_name.items()
])


df_mesh_lvl1.to_csv("mesh_lvl1_disease_mapping.csv", index=False, encoding="utf-8")

In [ ]:
df_plot = df_1.copy()


df_plot = df_plot.dropna(subset=["year"])


yearly_counts = df_plot.groupby(["year", "inst_type"]).size().reset_index(name="count")


yearly_counts["percentage"] = yearly_counts.groupby("year")["count"].transform(lambda x: x / x.sum() * 100)

In [ ]:
df_exploded = df_1.explode("mesh_lvl1")


df_exploded = df_exploded.dropna(subset=["mesh_lvl1", "inst_type_v2"])


count_table = (
    df_exploded
    .groupby(["mesh_lvl1", "inst_type"])
    .size()
    .reset_index(name="count")
)


pivot_table = count_table.pivot(
    index="mesh_lvl1",
    columns="inst_type",
    values="count"
).fillna(0).astype(int)

In [ ]:
df_disease = pivot_table.reset_index()
df_final = df_disease.merge(df_mesh_lvl1.rename(columns={'category_name':'disease_name'}), on="mesh_lvl1", how="left")
cols = ["mesh_lvl1", "disease_name"] + [c for c in df_final.columns if c not in ["mesh_lvl1", "disease_name"]]
df_final = df_final[cols]
df_final.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# =========================
# 1. Basic settings
# =========================

label_map = {
    "public": "Public",
    "private": "Non-public",
    "public+private": "Mixed"
}

color_map = {
    "Public": "#1f77b4",
    "Mixed": "#ff7f0e",
    "Non-public": "#2ca02c"
}

order = ["Public", "Mixed", "Non-public"]

# Overall funding distribution
overall_pct = {
    "Public": 53.36,
    "Mixed": 30.95,
    "Non-public": 15.70
}


# =========================
# 2. Panel A data
#    Temporal percentages
# =========================

yearly_plot = yearly_counts.copy()

yearly_plot["funding_structure"] = (
    yearly_plot["inst_type"].map(label_map)
)

perc_pivot = yearly_plot.pivot(
    index="year",
    columns="funding_structure",
    values="percentage"
).fillna(0)

perc_pivot = perc_pivot[order]


# =========================
# 3. Panel B data
#    Disease-area composition
# =========================

disease_plot = df_final.copy()

# Remove Animal Diseases
disease_plot = disease_plot[
    disease_plot["mesh_lvl1"] != "C22"
].copy()

# Rename funding columns
disease_plot = disease_plot.rename(columns={
    "public": "Public",
    "public+private": "Mixed",
    "private": "Non-public"
})

# Total publications
disease_plot["Total"] = (
    disease_plot["Public"]
    + disease_plot["Mixed"]
    + disease_plot["Non-public"]
)

# Keep original ordering:
# descending by number of publications
disease_plot = disease_plot.sort_values(
    "Total",
    ascending=False
).reset_index(drop=True)

# Percentages
for col in order:
    disease_plot[f"{col}_pct"] = (
        disease_plot[col]
        / disease_plot["Total"]
        * 100
    )

# Disease label + N
disease_plot["label"] = disease_plot["disease_name"]+ " (N="+ disease_plot["Total"].map(lambda x: f"{x:,}")+ ")"
#disease_plot["label"] = disease_plot["disease_name"]


# Wrap long disease names
disease_plot["label"] = (
    disease_plot["label"]
    .str.replace(
        "Congenital, Hereditary, and Neonatal Diseases and Abnormalities",
        "Congenital, Hereditary, and Neonatal Diseases\nand Abnormalities",
        regex=False
    )
    .str.replace(
        "Pathological Conditions, Signs and Symptoms",
        "Pathological Conditions, Signs\nand Symptoms",
        regex=False
    )
)

# =========================
# 4. Figure layout
# =========================

fig = plt.figure(figsize=(18, 11))


# [left, bottom, width, height]
ax1 = fig.add_axes([0.06, 0.34, 0.31, 0.40])
ax2 = fig.add_axes([0.59, 0.14, 0.38, 0.78])


# =========================
# 5. Panel A
#    Temporal shift
# =========================

for col in order:
    ax1.plot(
        perc_pivot.index,
        perc_pivot[col],
        marker="o",
        markersize=6,
        linewidth=2,
        label=col,
        color=color_map[col],
        alpha=0.75
    )

ax1.set_title(
    "(A) Funding Structure Over Time",
    fontsize=16,
    pad=12
)

ax1.set_xlabel(
    "Year",
    fontsize=15
)

ax1.set_ylabel(
    "Percentage of Publications (%)",
    fontsize=15
)

ax1.set_xticks([2010, 2015, 2020])

ax1.tick_params(
    axis="both",
    labelsize=13
)

ax1.grid(
    True,
    alpha=0.3
)

ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)


ax1.legend(
    title="Funding Structure",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.20),
    ncol=3,
    frameon=True,
    fontsize=12,
    title_fontsize=12
)


# =========================
# 6. Panel B
#    Disease-area composition
# =========================

y = range(len(disease_plot))

for col in order:
    ax2.scatter(
        disease_plot[f"{col}_pct"],
        y,
        s=55,
        color=color_map[col],
        alpha=0.75,
        zorder=3
    )

# Overall average reference lines
for col in order:
    ax2.axvline(
        overall_pct[col],
        color=color_map[col],
        linestyle="--",
        linewidth=1.5,
        alpha=0.45,
        zorder=1
    )

ax2.set_yticks(list(y))

ax2.set_yticklabels(
    disease_plot["label"],
    fontsize=11
)

ax2.invert_yaxis()

ax2.set_xlim(0, 80)
ax2.set_xticks(range(0, 81, 10))

ax2.set_xlabel(
    "Percentage of Publications (%)",
    fontsize=15
)

ax2.set_ylabel("")

ax2.tick_params(
    axis="x",
    labelsize=13
)

ax2.set_title(
    "(B) Funding Structure Across Disease Areas",
    fontsize=16,
    pad=12
)

ax2.grid(
    axis="y",
    alpha=0.25
)

ax2.grid(
    axis="x",
    linestyle="--",
    alpha=0.2
)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)


# =========================
# 7. Panel B legend
# =========================

legend_handles_b = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        markersize=8,
        markerfacecolor=color_map["Public"],
        markeredgecolor=color_map["Public"],
        alpha=0.75,
        label="Public"
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        markersize=8,
        markerfacecolor=color_map["Mixed"],
        markeredgecolor=color_map["Mixed"],
        alpha=0.75,
        label="Mixed"
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        markersize=8,
        markerfacecolor=color_map["Non-public"],
        markeredgecolor=color_map["Non-public"],
        alpha=0.75,
        label="Non-public"
    ),
    Line2D(
        [0], [0],
        color=color_map["Public"],
        linestyle="--",
        linewidth=1.5,
        alpha=0.5,
        label="Overall Public (53.36%)"
    ),
    Line2D(
        [0], [0],
        color=color_map["Mixed"],
        linestyle="--",
        linewidth=1.5,
        alpha=0.5,
        label="Overall Mixed (30.95%)"
    ),
    Line2D(
        [0], [0],
        color=color_map["Non-public"],
        linestyle="--",
        linewidth=1.5,
        alpha=0.5,
        label="Overall Non-public (15.70%)"
    )
]

ax2.legend(
    handles=legend_handles_b,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.075),
    ncol=3,
    frameon=True,
    fontsize=11
)

plt.show()

## Figure 3: impact patterns across disease areas

Read disease-specific model outputs and arrange coefficients and significance indicators by impact dimension and MeSH disease category. The model output files themselves are not distributed.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle

academic = pd.read_csv("a5_academic_disease_results_top10_logit_v2.csv").rename(
    columns={"Disease_Name": "disease_name", "Comparison Model": "comparison"}
)
policy = pd.read_csv("a5_policy_disease_result_final.csv")
patent = pd.read_csv("a5_patent_disease_result_final.csv")
clinical = pd.read_csv("a5_clinical_disease_result_final.csv")

# MeSH C/F order
mesh_order = [
    ("C01", "Infections"),
    ("C04", "Neoplasms"),
    ("C05", "Musculoskeletal Diseases"),
    ("C06", "Digestive System Diseases"),
    ("C07", "Stomatognathic Diseases"),
    ("C08", "Respiratory Tract Diseases"),
    ("C09", "Otorhinolaryngologic Diseases"),
    ("C10", "Nervous System Diseases"),
    ("C11", "Eye Diseases"),
    ("C12", "Urogenital Diseases"),
    ("C14", "Cardiovascular Diseases"),
    ("C15", "Hemic and Lymphatic Diseases"),
    ("C16", "Congenital, Hereditary, and Neonatal Diseases and Abnormalities"),
    ("C17", "Skin and Connective Tissue Diseases"),
    ("C18", "Nutritional and Metabolic Diseases"),
    ("C19", "Endocrine System Diseases"),
    ("C20", "Immune System Diseases"),
    ("C23", "Pathological Conditions, Signs and Symptoms"),
    ("C24", "Occupational Diseases"),
    ("C25", "Chemically-Induced Disorders"),
    ("C26", "Wounds and Injuries"),
    ("F03", "Mental Disorders")
]

order = [name for code, name in mesh_order]
mesh_codes = {name: code for code, name in mesh_order}

def prepare(df):
    rows = []

    for d in order:
        x = df[df["disease_name"] == d]

        mp = x[x["comparison"] == "mixed vs public"]
        pn = x[x["comparison"] == "public vs non-public"]
        mn = x[x["comparison"] == "mixed vs non-public"]

        rows.append([
            d,
            mp["coef"].iloc[0], mp["p_value"].iloc[0],
            -pn["coef"].iloc[0], pn["p_value"].iloc[0],
            mn["coef"].iloc[0], mn["p_value"].iloc[0]
        ])

    return pd.DataFrame(
        rows,
        columns=[
            "disease",
            "Mixed vs. public", "p1",
            "Non-public vs. public", "p2",
            "Mixed vs. non-public", "p3"
        ]
    )

dfs = [
    prepare(academic),
    prepare(policy),
    prepare(patent),
    prepare(clinical)
]

titles = ["Academic", "Policy", "Technological", "Clinical"]

overall = [
    [0.39, 0.001, 0.30, 0.001, 0.09, 0.001],
    [-0.05, 0.092, 0.17, 0.001, -0.22, 0.001],
    [0.12, 0.001, 0.06, 0.058, 0.06, 0.053],
    [0.05, 0.040, 0.27, 0.001, -0.22, 0.001]
]

cols = [
    "Mixed vs. public",
    "Non-public vs. public",
    "Mixed vs. non-public"
]

pcols = ["p1", "p2", "p3"]

comparisons = [
    "Mixed vs.\npublic",
    "Non-public\nvs. public",
    "Mixed vs.\nnon-public"
]

norm = TwoSlopeNorm(vmin=-0.75, vcenter=0, vmax=0.75)
cmap = plt.cm.RdBu

fig, axes = plt.subplots(
    1, 4,
    figsize=(17, 11),
    gridspec_kw={"wspace": 0.08}
)

for k, (ax, df, title) in enumerate(zip(axes, dfs, titles)):

    vals = df[cols].values
    ps = df[pcols].values

    ax.imshow(
        vals,
        cmap=cmap,
        norm=norm,
        aspect="auto"
    )

    for i in range(len(order)):
        for j in range(3):

            # Strongly fade all non-significant cells
            if ps[i,j] >= .05:
                ax.add_patch(Rectangle(
                    (j-.5, i-.5), 1, 1,
                    facecolor="white",
                    alpha=.55,
                    edgecolor="0.65",
                    linewidth=0,
                    hatch="///"
                ))

            ax.text(
                j,
                i,
                f"{vals[i, j]:+.2f}",
                ha="center",
                va="center",
                fontsize=8,
                fontweight="bold" if ps[i, j] < 0.05 else "normal",
                color="black" if ps[i, j] < 0.05 else "0.65"
            )

    # Number of disease areas with significant positive/negative coefficients
    sigtxt = []

    for col, pcol in zip(cols, pcols):
        pos = ((df[pcol] < 0.05) & (df[col] > 0)).sum()
        neg = ((df[pcol] < 0.05) & (df[col] < 0)).sum()
        sigtxt.append(f"+{pos} / −{neg}")

    # Comparison labels
    for j, label in enumerate(comparisons):
        ax.text(
            j,
            -3.05,
            label,
            ha="center",
            va="center",
            fontsize=10
        )

    # Impact title
    ax.text(
        1,
        -2.05,
        title,
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold"
    )

    ax.plot(
        [-0.45, 2.45],
        [-1.55, -1.55],
        color="0.35",
        linewidth=0.8,
        clip_on=False
    )

    if k == 0:
        ax.text(
            -0.55,
            -1.05,
            "Sig. areas (+/−):",
            ha="right",
            va="center",
            fontsize=10,
            color="0.4"
        )

    for j, s in enumerate(sigtxt):
        ax.text(
            j,
            -1.05,
            s,
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
            color="#2878B5" if j != 2 else "#C44E52"
        )

    ax.set_xticks([])
    ax.set_yticks(range(len(order)))

    if k == 0:
        labels = [
            f"{mesh_codes[d]}-{d}"
            for d in order
        ]

        labels = [
            x.replace(
                "Pathological Conditions, Signs and Symptoms",
                "Pathological Conditions,\nSigns and Symptoms"
            ).replace(
                "Congenital, Hereditary, and Neonatal Diseases and Abnormalities",
                "Congenital, Hereditary, and Neonatal\nDiseases and Abnormalities"
            )
            for x in labels
        ]

        ax.set_yticklabels(labels, fontsize=10)
    else:
        ax.set_yticklabels([])

    ax.tick_params(axis="y", length=0)

    ax.set_xlim(-0.5, 2.5)
    ax.set_ylim(len(order) - 0.5, -3.55)

    for s in ax.spines.values():
        s.set_visible(False)

plt.subplots_adjust(
    bottom=0.18,
    top=0.94,
    left=0.30,
    right=0.98
)

# Overall pooled models
for k, (ax, ov) in enumerate(zip(axes, overall)):

    pos = ax.get_position()

    oax = fig.add_axes([
        pos.x0,
        pos.y0 - 0.052,
        pos.width,
        0.035
    ])

    vals = np.array(ov[::2]).reshape(1, 3)
    ps = ov[1::2]

    oax.imshow(
        vals,
        cmap=cmap,
        norm=norm,
        aspect="auto"
    )

    for j, x in enumerate(vals[0]):

        if ps[j] >= .05:
            oax.add_patch(Rectangle(
                (j-.5, -.5), 1, 1,
                facecolor="white",
                alpha=.55,
                edgecolor="0.65",
                linewidth=0,
                hatch="///"
            ))
        else:
            oax.add_patch(Rectangle(
                (j-.5, -.5), 1, 1,
                fill=False,
                edgecolor="0.25",
                linewidth=1
            ))

        oax.text(
            j,
            0,
            f"{x:+.2f}",
            ha="center",
            va="center",
            fontsize=8,
            fontweight="bold" if ps[j] < 0.05 else "normal",
            color="black" if ps[j] < 0.05 else "0.65"
        )

    oax.set_xticks([])
    oax.set_yticks([])

    for s in oax.spines.values():
        s.set_visible(False)

    if k == 0:
        oax.text(
            -0.65,
            0,
            "Overall ",
            ha="right",
            va="center",
            fontsize=10,
            fontweight="bold"
        )

# Colorbar
cax = fig.add_axes([0.42, 0.072, 0.30, 0.012])

cb = fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=cax,
    orientation="horizontal"
)

cb.set_label(
    "Coefficient (log-odds): positive = first group higher, negative = second group higher",
    fontsize=10
)

cb.ax.tick_params(labelsize=10)

fig.text(
    0.57,
    -0.01,
    "Hatched cells indicate p ≥ 0.05; unhatched cells indicate p < 0.05.\n"
    "Academic = top-10% citation logit. Overall row reports regression results for the full sample.",
    ha="center",
    va="bottom",
    fontsize=10,
    color="0.4"
)


plt.show()

## Figure 4A: research orientation

Assign fractions of each paper's MeSH categories to human, animal, and cell or molecular research, then visualize the three funding groups in a triangular plot.


In [ ]:

mol_cell_codes = {
    "C05",  # Musculoskeletal Diseases
    "C06",  # Digestive System Diseases
    "C17",  # Skin and Connective Tissue Diseases
    "C18",  # Nutritional and Metabolic Diseases
    "C19",  # Endocrine System Diseases
    "C25",  # Chemically-Induced Disorders
}  

animal_codes = {
    "C22",  # Animal Diseases
}  
human_codes = {
    "C01",  # Infections
    "C04",  # Neoplasms
    "C07",  # Stomatognathic Diseases
    "C08",  # Respiratory Tract Diseases
    "C09",  # Otorhinolaryngologic Diseases
    "C10",  # Nervous System Diseases
    "C11",  # Eye Diseases
    "C12",  # Urogenital Diseases
    "C14",  # Cardiovascular Diseases
    "C15",  # Hemic and Lymphatic Diseases
    "C16",  # Congenital, Hereditary, and Neonatal Diseases and Abnormalities
    "C20",  # Immune System Diseases
    "C21",  # Disorders of Environmental Origin
    "C23",  # Pathological Conditions, Signs and Symptoms
    "C24",  # Occupational Diseases
    "C26",  # Wounds and Injuries
    "F03",  # Mental Disorders
}  


def assign_fraction(lst):
    n = len(lst)
    if n == 0:
        return (0, 0, 0)
    mol, ani, hum = 0, 0, 0
    for x in lst:
        if x in mol_cell_codes:
            mol += 1 / n
        elif x in animal_codes:
            ani += 1 / n
        elif x in human_codes:
            hum += 1 / n
    return (mol, ani, hum)

df_1[["mol_cell", "animal", "human"]] = df_1["mesh_lvl1"].apply(assign_fraction).apply(pd.Series)
#tmp = df_1["mesh_lvl1"].apply(assign_fraction)
#print(tmp.apply(lambda x: None if x is None else len(x)).value_counts(dropna=False))

In [ ]:
df_1['orientation_sum'] = (
    df_1['mol_cell'] +
    df_1['animal'] +
    df_1['human']
)

df_1['orientation_sum'].describe()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.path import Path
from scipy.ndimage import gaussian_filter
from matplotlib.colors import Normalize


def prepare_ternary_density(
    df,
    col_human="human",
    col_animal="animal",
    col_cell="mol_cell",
    nbins=300,
    sigma=30.0
):
    h = np.sqrt(3)

    human = df[col_human].to_numpy(dtype=float)
    animal = df[col_animal].to_numpy(dtype=float)
    cell = df[col_cell].to_numpy(dtype=float)

    mask = (
        np.isfinite(human) &
        np.isfinite(animal) &
        np.isfinite(cell)
    )

    human = human[mask]
    animal = animal[mask]
    cell = cell[mask]

    x = animal - cell
    y = h * human

    H2d, xedges, yedges = np.histogram2d(
        x,
        y,
        bins=nbins,
        range=[[-1.0, 1.0], [0.0, h]]
    )

    # Normalize within each funding structure
    H2d = H2d / H2d.sum()

    Z = gaussian_filter(H2d, sigma=sigma)

    Xc = 0.5 * (xedges[:-1] + xedges[1:])
    Yc = 0.5 * (yedges[:-1] + yedges[1:])
    X, Y = np.meshgrid(Xc, Yc, indexing="ij")

    tri = np.array([
        [-1.0, 0.0],
        [1.0, 0.0],
        [0.0, h]
    ])

    path = Path(tri)
    pts = np.column_stack([X.ravel(), Y.ravel()])
    tri_mask = path.contains_points(pts).reshape(X.shape)

    Z_masked = Z.astype(float)
    Z_masked[~tri_mask] = np.nan

    # Centroid in ternary space
    mean_human = human.mean()
    mean_animal = animal.mean()
    mean_cell = cell.mean()

    centroid_x = mean_animal - mean_cell
    centroid_y = h * mean_human

    return {
        "X": X,
        "Y": Y,
        "Z": Z_masked,
        "mask": tri_mask,
        "tri": tri,
        "centroid": (centroid_x, centroid_y),
        "means": (mean_human, mean_animal, mean_cell),
        "n": len(human)
    }

In [ ]:
def create_figure4_layout(figsize=(17, 10)):
    fig = plt.figure(figsize=figsize)

    outer = fig.add_gridspec(
        nrows=2,
        ncols=2,
        height_ratios=[1.05, 1],
        width_ratios=[1, 1],
        hspace=0.28,
        wspace=0.25,
        left=0.05,
        right=0.97,
        top=0.95,
        bottom=0.07
    )

    # A: entire top row
    # Spacer columns explicitly separate the three triangles
    gs_A = outer[0, :].subgridspec(
        nrows=1,
        ncols=5,
        width_ratios=[1, 0.38, 1, 0.38, 1],
        wspace=0
    )

    ax_A1 = fig.add_subplot(gs_A[0, 0])
    ax_A2 = fig.add_subplot(gs_A[0, 2])
    ax_A3 = fig.add_subplot(gs_A[0, 4])

    # B and C: bottom row
    ax_B = fig.add_subplot(outer[1, 0])
    ax_C = fig.add_subplot(outer[1, 1])

    # Temporarily blank
    ax_B.axis("off")
    ax_C.axis("off")

    return fig, [ax_A1, ax_A2, ax_A3], ax_B, ax_C

In [ ]:
def plot_figure4A(
    axes,
    df,
    funding_col="inst_type",
    public_value="public",
    mixed_value="public+private",
    nonpublic_value="private",
    col_human="human",
    col_animal="animal",
    col_cell="mol_cell",
    nbins=300,
    sigma=30.0
):
    h = np.sqrt(3)

    groups = {
        "Public": df[df[funding_col] == public_value],
        "Mixed": df[df[funding_col] == mixed_value],
        "Non-public": df[df[funding_col] == nonpublic_value]
    }

    cmaps = {
        "Public": "Blues",
        "Mixed": "Oranges",
        "Non-public": "Greens"
    }

    centroid_colors = {
        "Public": "tab:blue",
        "Mixed": "tab:orange",
        "Non-public": "tab:green"
    }

    results = {}

    # ---------- Prepare densities ----------
    for name, group_df in groups.items():
        results[name] = prepare_ternary_density(
            group_df,
            col_human=col_human,
            col_animal=col_animal,
            col_cell=col_cell,
            nbins=nbins,
            sigma=sigma
        )

    # Shared density scale
    vmax = max(
        np.nanmax(results[name]["Z"])
        for name in results
    )

    levels = np.linspace(0, vmax, 11)
    norm = Normalize(vmin=0, vmax=vmax)

    # ---------- Draw three triangles ----------
    for ax, name in zip(axes, groups.keys()):

        result = results[name]

        ax.set_facecolor("white")

        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.contourf(
            result["X"],
            result["Y"],
            result["Z"],
            levels=levels,
            cmap=cmaps[name],
            norm=norm,
            extend="max"
        )

        # Triangle border
        ax.add_patch(
            Polygon(
                result["tri"],
                closed=True,
                fill=False,
                edgecolor="black",
                linewidth=1.5
            )
        )

        # ---------- Own centroid ----------
        cx, cy = result["centroid"]

        ax.scatter(
            cx,
            cy,
            marker="x",
            s=100,
            color="black",
            linewidth=2.2,
            zorder=10
        )

        # ---------- Other two centroids ----------
        # Draw after X so that highly overlapping centroids remain visible
        for other_name in groups.keys():

            if other_name == name:
                continue

            ox, oy = results[other_name]["centroid"]

            ax.scatter(
                ox,
                oy,
                s=30,
                color=centroid_colors[other_name],
                edgecolor="white",
                linewidth=0.7,
                zorder=12
            )

        # ---------- Funding structure title ----------
        ax.set_title(
            name,
            fontsize=14,
            fontweight="bold",
            color=centroid_colors[name],
            pad=10
        )

        # ---------- Triangle labels ----------
        ax.text(
            0,
            h + 0.055,
            "Human",
            ha="center",
            va="bottom",
            fontsize=11
        )

        ax.text(
            -1.04,
            -0.055,
            "Cell &\nMolecular",
            ha="right",
            va="top",
            fontsize=10.5,
            linespacing=0.9
        )

        ax.text(
            1.04,
            -0.055,
            "Animal",
            ha="left",
            va="top",
            fontsize=10.5
        )

        ax.set_aspect("equal")

        # Extra whitespace makes triangles smaller
        ax.set_xlim(-1.32, 1.32)
        ax.set_ylim(-0.22, h + 0.20)

        ax.set_xticks([])
        ax.set_yticks([])

    return results

In [ ]:
results_4A = plot_figure4A(
    #axes_A,
    df_1,
    funding_col="inst_type",
    public_value="public",
    mixed_value="public+private",
    nonpublic_value="private",
    col_human="human",
    col_animal="animal",
    col_cell="mol_cell",
    nbins=300,
    sigma=30.0
)

## Figure 4: funder subtypes and orientation controls

Separate non-public funders into company, university or hospital, and foundation or NGO groups, and specify additional models that examine research orientation and funding subtype.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


def orientation_ols_compare(
    df,
    funding_col='inst_type',
    outcomes=('human', 'animal', 'mol_cell')
):
    results = []
    model_stats = {}

    outcome_names = {
        'human': 'Human',
        'animal': 'Animal',
        'mol_cell': 'Cell & Molecular'
    }

    for outcome in outcomes:

        tmp = df[
            [funding_col, outcome]
        ].dropna().copy()

        tmp = tmp[
            tmp[funding_col].isin(
                ['public', 'public+private', 'private']
            )
        ]

        # Public = reference
        tmp['mixed'] = (
            tmp[funding_col] == 'public+private'
        ).astype(int)

        tmp['nonpublic'] = (
            tmp[funding_col] == 'private'
        ).astype(int)

        X = sm.add_constant(
            tmp[['mixed', 'nonpublic']]
        )

        y = tmp[outcome]

        model = sm.OLS(y, X).fit()

        orientation_name = outcome_names[outcome]

        # -------------------------------
        # Model statistics
        # -------------------------------
        model_stats[orientation_name] = {
            'R2': model.rsquared,
            'N': int(model.nobs)
        }

        # -------------------------------
        # Mixed vs Public
        # -------------------------------
        ci = model.conf_int().loc['mixed']

        results.append({
            'Orientation': orientation_name,
            'Comparison': 'Mixed vs Public',
            'Difference': model.params['mixed'],
            'SE': model.bse['mixed'],
            'CI_low': ci.iloc[0],
            'CI_high': ci.iloc[1],
            'p_value': model.pvalues['mixed']
        })

        # -------------------------------
        # Non-public vs Public
        # -------------------------------
        ci = model.conf_int().loc['nonpublic']

        results.append({
            'Orientation': orientation_name,
            'Comparison': 'Non-public vs Public',
            'Difference': model.params['nonpublic'],
            'SE': model.bse['nonpublic'],
            'CI_low': ci.iloc[0],
            'CI_high': ci.iloc[1],
            'p_value': model.pvalues['nonpublic']
        })

        # -------------------------------
        # Mixed vs Non-public
        # -------------------------------
        contrast = np.array([0, 1, -1])
        test = model.t_test(contrast)

        ci = test.conf_int()

        results.append({
            'Orientation': orientation_name,
            'Comparison': 'Mixed vs Non-public',
            'Difference': test.effect.item(),
            'SE': test.sd.item(),
            'CI_low': ci[0, 0].item(),
            'CI_high': ci[0, 1].item(),
            'p_value': np.asarray(test.pvalue).item()
        })

    results = pd.DataFrame(results)

    return results, model_stats

In [ ]:
df_5 = pd.read_csv('a1_agency_label.csv').rename(columns={'Unnamed: 0': 'ID'})
df_5

In [ ]:
category_to_bit = {
    'Public Agency': 1,
    'Private Company': 2,
    'University': 4,
    'Other Organization': 8
}

id_to_bit = dict(zip(
    df_5['ID'],
    df_5['category'].map(category_to_bit)
))

def get_funding_bits(ids):
    bits = 0
    for i in ids:
        bits |= id_to_bit.get(i, 0)
    return bits

funding_bits = df_1['matched_IDs'].apply(get_funding_bits)

df_1['has_public'] = (funding_bits & 1) > 0
df_1['has_company'] = (funding_bits & 2) > 0
df_1['has_university'] = (funding_bits & 4) > 0
df_1['has_other'] = (funding_bits & 8) > 0

In [ ]:
def label_nonpublic_combination(row):
    types = []

    if row['has_company']:
        types.append('Company')

    if row['has_university']:
        types.append('University')

    if row['has_other']:
        types.append('Other')

    return ' + '.join(types) if types else 'None'


combination_summary['Combination'] = combination_summary.apply(
    label_nonpublic_combination,
    axis=1
)

combination_summary = combination_summary[
    ['Combination', 'N', 'Percent']
].sort_values('N', ascending=False)

combination_summary

In [ ]:
def assign_funding_subtype(row):
    # Public-only
    if row['inst_type'] == 'public':
        return 'Public'

    # Mixed public + non-public
    if row['inst_type'] == 'public+private':
        return 'Mixed'

    # Non-public-only
    types = []

    if row['has_company']:
        types.append('Private company')

    if row['has_university']:
        types.append('University / hospital')

    if row['has_other']:
        types.append('Foundation / NGO')

    if len(types) == 1:
        return types[0]
    elif len(types) > 1:
        return 'Multiple non-public'

    return np.nan


df_1['funding_subtype'] = df_1.apply(
    assign_funding_subtype,
    axis=1
)

df_1['funding_subtype'].value_counts(dropna=False)

In [ ]:
import statsmodels.api as sm
import patsy
formula = "top10_pct ~ C(funding_subtype, Treatment(reference='Public'))+ C(year)+C(q_num)+C(team_size_group)+C(document_type_group)"  

y, X = patsy.dmatrices(formula, df_citation_j3, return_type="dataframe")

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
)

res = model.fit()
print(res.summary())

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(funding_subtype, Treatment(reference="Public"))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_p,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_p["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(funding_subtype, Treatment(reference="Public"))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_pp,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_pp["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(funding_subtype, Treatment(reference='Public'))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_c,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_c["weight"].astype(float)
)

res = model.fit()
print(res.summary())